# 🤸 Humanoid Backflip — PPO vs DQN Baselines

Train a MuJoCo Humanoid to do a backflip using two methods:
1. **PPO** (continuous actions) — baseline
2. **DQN** (discretised actions) — baseline

Reward = jump shaping + backflip-specific:
- **Jump rewards:** upward velocity, flight, crouch, liftoff, height, posture, drift, jerk, control
- **Backflip rewards:** pitch angular velocity, foot height (inverted body), rotation progress, tuck, landing quality, rotation completion, off-axis penalty

Rewards are split into two composable functions (`compute_jump_rewards` / `compute_backflip_rewards`) so each can be modified independently.

**Runtime → Change to GPU (T4) for faster training.**

In [1]:
# @title 1. Environment Setup & Installation

!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common \
    patchelf \
    ffmpeg \
    xvfb

!pip install gymnasium[mujoco] stable-baselines3 shimmy imageio[ffmpeg] pyvirtualdisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()
print('✅ Virtual display started')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
software-properties-common is already the newest version (0.99.22.9).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
The following additional packages will be installed:
  libegl-dev libgl-dev libgles-dev libgles1 libglew2.2 libglu1-mesa
  libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev libopengl-dev
  libosmesa6
Suggested packages:
  glew-utils
The following NEW packages will be installed:
  libegl-dev libgl-dev libgl1-mesa-dev libgles-dev libgles1 libglew-dev
  libglew2.2 libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev
  libglx-dev libopengl-dev libosmesa6 libosmesa6-dev patchelf
0 upgraded, 16 newly installed, 0 to remove and 37 not upgraded.
Need to get 4,275 kB of archives.
After this operation, 20.5 MB of 

In [2]:
# @title 2. Backflip Environment + Reward Functions

import gymnasium as gym
import numpy as np
from gymnasium import spaces

# ── Constants ──────────────────────────────────────────────────────
FLOOR_GEOM_ID      = 0
RIGHT_FOOT_GEOM_ID = 8
LEFT_FOOT_GEOM_ID  = 11
FOOT_GEOM_IDS      = {RIGHT_FOOT_GEOM_ID, LEFT_FOOT_GEOM_ID}
RIGHT_FOOT_BODY    = 'foot_right'
LEFT_FOOT_BODY     = 'foot_left'
STANDING_Z          = 1.4


def quat_to_euler(q):
    w, x, y, z = q
    roll  = np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    pitch = np.arcsin(np.clip(2*(w*y - z*x), -1, 1))
    yaw   = np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    return roll, pitch, yaw

def quat_to_pitch(q):
    w, x, y, z = q
    return np.arcsin(np.clip(2*(w*y - z*x), -1, 1))


# ═══════════════════════════════════════════════════════════════════
#  JUMP REWARDS (composable)
# ═══════════════════════════════════════════════════════════════════
def compute_jump_rewards(z, vz, airborne, grounded, body_fell, x, y,
                         action, action_hist, current_jump_steps,
                         vz_at_liftoff, min_z_this_attempt):
    bd = {}; reward = 0.0

    r = 0.5 * np.clip(vz, 0.0, 5.0)
    reward += r; bd['velocity'] = r

    r = 0.0
    if airborne and z > 0.8:
        ha = max(0.0, z - STANDING_Z)
        r = 1.0 + 3.0*ha + 8.0*ha**2 + np.clip(vz, 0.0, 5.0)
    reward += r; bd['flight'] = r

    r = 0.2*vz_at_liftoff**2 if (airborne and z > 0.8 and current_jump_steps == 1) else 0.0
    reward += r; bd['liftoff'] = r

    r = 0.3*min(STANDING_Z-z, 0.3) if (grounded and z < STANDING_Z-0.05 and z > 0.8) else 0.0
    reward += r; bd['crouch'] = r

    r = 0.2*max(0.0, z-1.0)
    reward += r; bd['height'] = r

    r = -3.0 if body_fell else 0.0
    reward += r; bd['posture'] = r

    drift = np.sqrt(x**2 + y**2)
    r = -0.1*drift
    reward += r; bd['drift'] = r

    jerk = action - 2.0*action_hist[-1] + action_hist[-2]
    r = -0.02*np.sum(np.square(jerk))
    reward += r; bd['jerk'] = r

    r = -0.005*np.sum(np.square(action))
    reward += r; bd['control'] = r

    return reward, bd


# ═══════════════════════════════════════════════════════════════════
#  BACKFLIP REWARDS (composable)
# ═══════════════════════════════════════════════════════════════════
def compute_backflip_rewards(z, airborne, quat, angular_vel_y,
                             cumulative_pitch, foot_max_z,
                             grounded, body_fell,
                             landed_after_flip, rotation_complete):
    bd = {}; reward = 0.0

    # 1. Pitch angular velocity (backward rotation)
    r = 2.0*np.clip(angular_vel_y, 0.0, 8.0) if (airborne and z > 0.8) else 0.0
    reward += r; bd['ang_velocity'] = r

    # 2. Foot height (feet above torso → inverted body)
    r = 0.0
    if airborne and z > 0.8:
        r = 5.0*max(0.0, foot_max_z - z) + 1.0*max(0.0, foot_max_z - STANDING_Z)
    reward += r; bd['foot_height'] = r

    # 3. Rotation progress toward 2π
    r = 0.0
    if airborne:
        frac = min(abs(cumulative_pitch)/(2*np.pi), 1.0)
        r = 3.0*frac
        if frac > 0.75: r += 5.0*(frac - 0.75)
    reward += r; bd['rotation'] = r

    # 4. Tuck bonus
    r = 0.0
    if airborne and z > 0.8 and abs(cumulative_pitch) > 0.5:
        r = 0.5*min(max(0.0, STANDING_Z - z + 0.2), 0.4)
    reward += r; bd['tuck'] = r

    # 5. Landing bonus (upright after ≥80% rotation)
    r = 0.0
    if landed_after_flip and not body_fell:
        rot_frac = abs(cumulative_pitch)/(2*np.pi)
        if rot_frac > 0.8:
            _, pitch, _ = quat_to_euler(quat)
            r = 20.0*max(0.0, np.cos(pitch))*min(rot_frac, 1.2)
    reward += r; bd['landing'] = r

    # 6. Rotation completion (one-time)
    r = 15.0 if rotation_complete else 0.0
    reward += r; bd['completion'] = r

    # 7. Off-axis penalty
    r = 0.0
    if airborne:
        roll, _, yaw = quat_to_euler(quat)
        r = -0.5*(abs(roll) + abs(yaw))
    reward += r; bd['off_axis'] = r

    return reward, bd


# ═══════════════════════════════════════════════════════════════════
#  BACKFLIP REWARD WRAPPER
# ═══════════════════════════════════════════════════════════════════
class BackflipRewardWrapper(gym.Wrapper):
    def __init__(self, env, max_episode_steps=500,
                 jump_reward_scale=1.0, backflip_reward_scale=1.0):
        super().__init__(env)
        self.max_episode_steps = max_episode_steps
        self.jump_reward_scale = jump_reward_scale
        self.backflip_reward_scale = backflip_reward_scale
        self._reset_tracking()

    def _reset_tracking(self):
        self.step_count = 0
        self.max_z = -np.inf
        self.total_flight_steps = 0
        self.max_flight_z = -np.inf
        self.entered_flight = False
        self.was_crouching = False
        self.min_z_this_attempt = STANDING_Z
        self.was_grounded = True
        self.vz_at_liftoff = 0.0
        self.current_jump_steps = 0
        self.action_hist = [np.zeros(self.action_space.shape),
                            np.zeros(self.action_space.shape)]
        self.cumulative_pitch = 0.0
        self.prev_pitch = 0.0
        self.max_foot_z = 0.0
        self.was_in_flight = False
        self.rotation_complete_given = False
        self.best_rotation = 0.0
        self.last_reward_breakdown = {}

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._reset_tracking()
        return obs, info

    def _get_floor_contacts(self):
        d = self.unwrapped.data
        t = set()
        for i in range(d.ncon):
            c = d.contact[i]; pair = {c.geom1, c.geom2}
            if FLOOR_GEOM_ID in pair: t.add((pair - {FLOOR_GEOM_ID}).pop())
        return t

    def _feet_on_ground(self, fc): return len(FOOT_GEOM_IDS & fc)
    def _non_foot_on_ground(self, fc): return len(fc - FOOT_GEOM_IDS) > 0

    def _get_foot_positions(self):
        d, m = self.unwrapped.data, self.unwrapped.model
        try:
            rf_z = d.xpos[m.body(RIGHT_FOOT_BODY).id][2]
            lf_z = d.xpos[m.body(LEFT_FOOT_BODY).id][2]
        except Exception:
            rf_z = d.geom_xpos[RIGHT_FOOT_GEOM_ID][2]
            lf_z = d.geom_xpos[LEFT_FOOT_GEOM_ID][2]
        return rf_z, lf_z

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        self.step_count += 1

        x, y, z = self.unwrapped.data.qpos[0:3]
        vx, vy, vz = self.unwrapped.data.qvel[0:3]
        quat = self.unwrapped.data.qpos[3:7]
        ang_vel = self.unwrapped.data.qvel[3:6]
        ang_vel_y = ang_vel[1]

        fc = self._get_floor_contacts()
        n_feet = self._feet_on_ground(fc)
        body_fell = self._non_foot_on_ground(fc)
        airborne = (n_feet == 0) and not body_fell
        grounded = (n_feet > 0)
        self.max_z = max(self.max_z, z)

        # Jump tracking
        if grounded:
            self.min_z_this_attempt = min(self.min_z_this_attempt, z)
            if z < STANDING_Z - 0.1: self.was_crouching = True
            if not self.was_grounded:
                self.current_jump_steps = 0
                self.was_crouching = False
                self.min_z_this_attempt = z
        if self.was_grounded and not grounded and not body_fell:
            self.vz_at_liftoff = max(vz, 0.0)
            self.current_jump_steps = 0
            self.cumulative_pitch = 0.0
            self.prev_pitch = quat_to_pitch(quat)
            self.rotation_complete_given = False
        self.was_grounded = grounded

        if airborne and z > 0.8:
            self.current_jump_steps += 1
            self.total_flight_steps += 1
            self.max_flight_z = max(self.max_flight_z, z)
            self.entered_flight = True

        # Rotation tracking
        if airborne:
            dt = self.unwrapped.dt  # Gymnasium natively handles timestep * frame_skip here
            self.cumulative_pitch += ang_vel_y * dt
        self.prev_pitch = quat_to_pitch(quat)

        # Foot height
        rf_z, lf_z = self._get_foot_positions()
        foot_max_z = max(rf_z, lf_z)
        self.max_foot_z = max(self.max_foot_z, foot_max_z)

        # Landing & rotation
        landed = self.was_in_flight and grounded and not body_fell
        self.was_in_flight = airborne
        rot_complete = False
        if abs(self.cumulative_pitch) > 1.8*np.pi and not self.rotation_complete_given:
            self.rotation_complete_given = True; rot_complete = True
        self.best_rotation = max(self.best_rotation, abs(self.cumulative_pitch))

        # ── Compute rewards ──
        r_j, bd_j = compute_jump_rewards(
            z, vz, airborne, grounded, body_fell, x, y,
            action, self.action_hist, self.current_jump_steps,
            self.vz_at_liftoff, self.min_z_this_attempt)
        r_f, bd_f = compute_backflip_rewards(
            z, airborne, quat, ang_vel_y, self.cumulative_pitch,
            foot_max_z, grounded, body_fell, landed, rot_complete)

        reward = self.jump_reward_scale*r_j + self.backflip_reward_scale*r_f

        bd = {}; bd.update(bd_j)
        bd.update({f'flip_{k}': v for k, v in bd_f.items()})
        bd['jump_subtotal'] = r_j; bd['flip_subtotal'] = r_f; bd['total'] = reward
        self.last_reward_breakdown = bd

        drift = np.sqrt(x**2 + y**2)
        done = terminated or truncated or self.step_count >= self.max_episode_steps
        if done:
            if self.entered_flight:
                info['jump_height'] = self.max_flight_z - STANDING_Z
                info['flight_steps'] = self.total_flight_steps
                info['best_rotation_deg'] = np.degrees(self.best_rotation)
                info['max_foot_z'] = self.max_foot_z
            if not terminated and not truncated: truncated = True

        if z < 0.2: terminated = True; reward -= 3.0
        if drift > 2.0: terminated = True; reward -= 2.0

        self.action_hist[-2] = self.action_hist[-1].copy()
        self.action_hist[-1] = action.copy()

        info.update({'z': z, 'vz': vz, 'airborne': airborne, 'drift': drift,
                     'cumulative_pitch': self.cumulative_pitch,
                     'cumulative_pitch_deg': np.degrees(self.cumulative_pitch),
                     'foot_max_z': foot_max_z, 'angular_vel_y': ang_vel_y,
                     'reward_breakdown': bd})
        return obs, reward, terminated, truncated, info


# ═══════════════════════════════════════════════════════════════════
#  DISCRETE ACTION WRAPPER (for DQN)
# ═══════════════════════════════════════════════════════════════════
class DiscreteActionWrapper(gym.ActionWrapper):
    def __init__(self, env, n_actions=64, seed=42):
        super().__init__(env)
        self.n_actions = n_actions
        self.continuous_space = env.action_space
        self.action_space = spaces.Discrete(n_actions)
        rng = np.random.RandomState(seed)
        low, high = self.continuous_space.low, self.continuous_space.high
        dim = self.continuous_space.shape[0]
        prims = [np.zeros(dim), .5*np.ones(dim), -.5*np.ones(dim),
                 np.ones(dim), -np.ones(dim)]
        for i in range(min(dim, 17)):
            vp = np.zeros(dim); vp[i] = 1.0
            vn = np.zeros(dim); vn[i] = -1.0
            prims.extend([vp, vn])
        n_rand = max(0, n_actions - len(prims))
        rands = rng.uniform(low, high, (n_rand, dim))
        all_a = prims[:n_actions]
        if len(all_a) < n_actions: all_a = prims + list(rands)
        self.codebook = np.clip(np.array(all_a[:n_actions], dtype=np.float32), low, high)

    def action(self, idx): return self.codebook[idx]


print('✅ Environment + reward functions defined')
print(f'   Jump reward components: velocity, flight, liftoff, crouch, height, posture, drift, jerk, control')
print(f'   Flip reward components: ang_velocity, foot_height, rotation, tuck, landing, completion, off_axis')

✅ Environment + reward functions defined
   Jump reward components: velocity, flight, liftoff, crouch, height, posture, drift, jerk, control
   Flip reward components: ang_velocity, foot_height, rotation, tuck, landing, completion, off_axis


---
## Part A: PPO Baseline

In [3]:
# @title 3. PPO Backflip Training

import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback


class BackflipCallback(BaseCallback):
    def __init__(self, tag='PPO', print_every=20_000, verbose=0):
        super().__init__(verbose)
        self.tag = tag; self.print_every = print_every
        self.heights = []; self.rotations = []; self.foot_zs = []
        self.ep_count = 0; self.flight_count = 0

    def _on_step(self):
        for info in self.locals.get('infos', []):
            self.ep_count += 1
            if 'jump_height' in info:
                self.heights.append(info['jump_height']); self.flight_count += 1
            if 'best_rotation_deg' in info: self.rotations.append(info['best_rotation_deg'])
            if 'max_foot_z' in info: self.foot_zs.append(info['max_foot_z'])
        if self.num_timesteps % self.print_every == 0:
            frac = self.flight_count / max(self.ep_count, 1)
            if self.rotations:
                rr, rh = self.rotations[-50:], self.heights[-50:] or [0]
                rf = self.foot_zs[-50:] or [0]
                print(f'  [{self.tag}] step={self.num_timesteps:,}  '
                      f'flight={frac:.1%}  avg_rot={np.mean(rr):.1f}°  '
                      f'max_rot={np.max(rr):.1f}°  avg_h={np.mean(rh):.3f}m  '
                      f'avg_foot_z={np.mean(rf):.2f}')
            else:
                print(f'  [{self.tag}] step={self.num_timesteps:,}  NO FLIGHTS ({self.ep_count} eps)')
        return True


# ── Config ──
N_ENVS = 4   # 4 for Colab (DummyVecEnv is sequential); use 8 with SubprocVecEnv locally
PPO_TIMESTEPS = 10_000_000   # Backflip needs more training than simple jump
MAX_EP_STEPS = 500
JUMP_SCALE = 1.0
FLIP_SCALE = 1.0

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


def make_ppo_env(seed=0):
    def _init():
        e = gym.make('Humanoid-v5', render_mode=None)
        e = BackflipRewardWrapper(e, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
        e.reset(seed=seed)
        return e
    return _init

# DummyVecEnv for Colab compatibility (SubprocVecEnv crashes in notebooks)
venv = DummyVecEnv([make_ppo_env(seed=i) for i in range(N_ENVS)])
ppo_env = VecNormalize(venv, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, clip_reward=10.0, gamma=0.99)

ppo_model = PPO(
    'MlpPolicy', ppo_env, verbose=1,
    learning_rate=3e-4, n_steps=2048, batch_size=1024, n_epochs=5,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2,
    ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, target_kl=0.05,
    device=device,
    policy_kwargs=dict(activation_fn=nn.ELU,
                       net_arch=dict(pi=[256, 256], vf=[256, 256])),
    tensorboard_log='./backflip_ppo_tb/',
)

print(f'\n--- PPO Backflip Training ({PPO_TIMESTEPS:,} steps) ---\n')
ppo_model.learn(total_timesteps=PPO_TIMESTEPS,
                callback=[BackflipCallback('PPO', 20_000)])

ppo_model.save('backflip_ppo')
ppo_env.save('backflip_ppo_vecnorm.pkl')
ppo_env.close()
print('\n✅ PPO saved: backflip_ppo.zip')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Device: cuda


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cuda device

--- PPO Backflip Training (10,000,000 steps) ---

Logging to ./backflip_ppo_tb/PPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Streaming output truncated to the last 5000 lines.
|    iterations           | 965        |
|    time_elapsed         | 17097      |
|    total_timesteps      | 7905280    |
| train/                  |            |
|    approx_kl            | 0.04664529 |
|    clip_fraction        | 0.399      |
|    clip_range           | 0.2        |
|    entropy_loss         | -36.3      |
|    explained_variance   | 1          |
|    learning_rate        | 0.0003     |
|    loss                 | -0.38      |
|    n_updates            | 4578       |
|    policy_gradient_loss | -0.0177    |
|    std                  | 2.16       |
|    value_loss           | 0.000477   |
----------------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 462        |
|    iterations           | 966        |
|    time_elapsed         | 17115      |
|    total_timesteps      | 7913472    |
| train/                  |            |
|    a

In [4]:
# @title 4. PPO Backflip Check

import os

def run_check(algo, max_steps=500, n_actions=64):
    is_dqn = (algo == 'dqn')
    model_path = f'backflip_{algo}'
    vn_path = f'backflip_{algo}_vecnorm.pkl'

    raw = gym.make('Humanoid-v5', render_mode=None)
    e = BackflipRewardWrapper(raw, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
    if is_dqn: e = DiscreteActionWrapper(e, n_actions=n_actions)

    def _mk():
        env = gym.make('Humanoid-v5', render_mode=None)
        env = BackflipRewardWrapper(env, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
        if is_dqn: env = DiscreteActionWrapper(env, n_actions=n_actions)
        return env
    dummy = DummyVecEnv([_mk])
    vec = VecNormalize.load(vn_path, dummy) if os.path.exists(vn_path) else None
    if vec: vec.training = False; vec.norm_reward = False

    from stable_baselines3 import DQN as _DQN
    model = (_DQN if is_dqn else PPO).load(model_path)
    obs, info = e.reset()
    cum = 0.0; heights = []; rotations = []

    for step in range(max_steps):
        obs_n = vec.normalize_obs(obs) if vec else obs
        act, _ = model.predict(obs_n, deterministic=True)
        obs, r, term, trunc, info = e.step(act)
        cum += r
        if step % 100 == 0:
            print(f'  step={step:>4}  z={info["z"]:.3f}  '
                  f'pitch={info["cumulative_pitch_deg"]:.1f}°  '
                  f'foot_z={info["foot_max_z"]:.2f}  r={r:.2f}  cum={cum:.1f}')
        if term or trunc:
            if 'jump_height' in info: heights.append(info['jump_height'])
            if 'best_rotation_deg' in info: rotations.append(info['best_rotation_deg'])
            print(f'  → Ep ended step {step}, cum={cum:.1f}, rot={info.get("best_rotation_deg",0):.1f}°')
            obs, info = e.reset(); cum = 0.0

    e.close()
    if vec: vec.close()
    return heights, rotations

print('--- PPO Backflip Check ---')
ppo_h, ppo_r = run_check('ppo')
if ppo_r: print(f'  Rotations: avg={np.mean(ppo_r):.1f}°  max={np.max(ppo_r):.1f}°')
if ppo_h: print(f'  Heights:   avg={np.mean(ppo_h):.3f}m  max={np.max(ppo_h):.3f}m')

--- PPO Backflip Check ---
  step=   0  z=1.404  pitch=-3.4°  foot_z=0.19  r=1.04  cum=1.0
  → Ep ended step 51, cum=675.9, rot=237.9°
  step= 100  z=1.118  pitch=217.6°  foot_z=0.26  r=16.94  cum=622.1
  → Ep ended step 103, cum=673.4, rot=238.9°
  → Ep ended step 155, cum=677.0, rot=240.1°
  step= 200  z=1.265  pitch=186.5°  foot_z=0.37  r=16.69  cum=555.7
  → Ep ended step 207, cum=672.7, rot=236.1°
  → Ep ended step 259, cum=673.6, rot=241.3°
  step= 300  z=1.334  pitch=157.1°  foot_z=0.44  r=16.63  cum=487.1
  → Ep ended step 311, cum=672.2, rot=240.7°
  → Ep ended step 362, cum=656.9, rot=237.2°
  step= 400  z=1.363  pitch=134.6°  foot_z=0.60  r=16.59  cum=441.6
  → Ep ended step 414, cum=676.1, rot=236.9°
  → Ep ended step 466, cum=676.3, rot=240.3°
  Rotations: avg=238.8°  max=241.3°
  Heights:   avg=0.040m  max=0.053m


---
## Part B: DQN Baseline

In [5]:
# @title 5. DQN Backflip Training

from stable_baselines3 import DQN

N_DISCRETE_ACTIONS = 64
DQN_TIMESTEPS = 5_000_000


def make_dqn_env(seed=0):
    def _init():
        e = gym.make('Humanoid-v5', render_mode=None)
        e = BackflipRewardWrapper(e, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
        e = DiscreteActionWrapper(e, n_actions=N_DISCRETE_ACTIONS, seed=seed)
        e.reset(seed=seed)
        return e
    return _init


dqn_venv = DummyVecEnv([make_dqn_env(seed=0)])
dqn_env = VecNormalize(dqn_venv, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, clip_reward=10.0, gamma=0.99)

dqn_model = DQN(
    'MlpPolicy', dqn_env, verbose=1,
    learning_rate=1e-4,
    buffer_size=200_000,
    learning_starts=10_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    exploration_fraction=0.3,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    device=device,
    policy_kwargs=dict(net_arch=[256, 256]),
    tensorboard_log='./backflip_dqn_tb/',
)

print(f'\n--- DQN Backflip Training ({DQN_TIMESTEPS:,} steps, {N_DISCRETE_ACTIONS} actions) ---\n')
dqn_model.learn(total_timesteps=DQN_TIMESTEPS,
                callback=[BackflipCallback('DQN', 20_000)])

dqn_model.save('backflip_dqn')
dqn_env.save('backflip_dqn_vecnorm.pkl')
dqn_env.close()
print('\n✅ DQN saved: backflip_dqn.zip')

Streaming output truncated to the last 5000 lines.
|    loss             | 0.000105 |
|    n_updates        | 1237032  |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 236900   |
|    fps              | 273      |
|    time_elapsed     | 18138    |
|    total_timesteps  | 4958241  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 9.79e-05 |
|    n_updates        | 1237060  |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 236904   |
|    fps              | 273      |
|    time_elapsed     | 18139    |
|    total_timesteps  | 4958351  |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 8.58e-05 |
|   

In [6]:
# @title 6. DQN Backflip Check

print('--- DQN Backflip Check ---')
dqn_h, dqn_r = run_check('dqn', n_actions=N_DISCRETE_ACTIONS)
if dqn_r: print(f'  Rotations: avg={np.mean(dqn_r):.1f}°  max={np.max(dqn_r):.1f}°')
if dqn_h: print(f'  Heights:   avg={np.mean(dqn_h):.3f}m  max={np.max(dqn_h):.3f}m')

--- DQN Backflip Check ---
  step=   0  z=1.399  pitch=-0.1°  foot_z=0.18  r=1.05  cum=1.0
  → Ep ended step 43, cum=93.0, rot=19.1°
  → Ep ended step 92, cum=41.8, rot=11.3°
  step= 100  z=1.283  pitch=-11.3°  foot_z=0.38  r=1.08  cum=8.2
  → Ep ended step 160, cum=43.3, rot=11.3°
  step= 200  z=1.134  pitch=10.4°  foot_z=0.12  r=0.07  cum=44.7
  → Ep ended step 213, cum=64.8, rot=10.4°
  → Ep ended step 266, cum=49.1, rot=11.5°
  step= 300  z=1.114  pitch=-5.3°  foot_z=0.29  r=0.08  cum=19.5
  → Ep ended step 329, cum=40.9, rot=20.7°
  → Ep ended step 394, cum=95.3, rot=14.1°
  step= 400  z=1.325  pitch=-8.9°  foot_z=0.32  r=1.11  cum=6.0
  → Ep ended step 447, cum=32.2, rot=10.5°
  → Ep ended step 487, cum=116.9, rot=38.8°
  Rotations: avg=16.4°  max=38.8°
  Heights:   avg=-0.002m  max=0.009m


---
## Part C: Comparison & Video

In [7]:
# @title 7. PPO vs DQN Comparison

def full_eval(algo, n_episodes=10, n_actions=64):
    is_dqn = (algo == 'dqn')
    vn_path = f'backflip_{algo}_vecnorm.pkl'

    raw = gym.make('Humanoid-v5', render_mode=None)
    e = BackflipRewardWrapper(raw, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
    if is_dqn: e = DiscreteActionWrapper(e, n_actions=n_actions)

    def _mk():
        env = gym.make('Humanoid-v5', render_mode=None)
        env = BackflipRewardWrapper(env, MAX_EP_STEPS, JUMP_SCALE, FLIP_SCALE)
        if is_dqn: env = DiscreteActionWrapper(env, n_actions=n_actions)
        return env
    dummy = DummyVecEnv([_mk])
    vec = VecNormalize.load(vn_path, dummy) if os.path.exists(vn_path) else None
    if vec: vec.training = False; vec.norm_reward = False

    model = (DQN if is_dqn else PPO).load(f'backflip_{algo}')
    heights, rotations, rewards = [], [], []

    for ep in range(n_episodes):
        obs, _ = e.reset(); cum = 0.0; done = False
        while not done:
            obs_n = vec.normalize_obs(obs) if vec else obs
            act, _ = model.predict(obs_n, deterministic=True)
            obs, r, term, trunc, info = e.step(act)
            cum += r; done = term or trunc
        rewards.append(cum)
        if 'jump_height' in info: heights.append(info['jump_height'])
        if 'best_rotation_deg' in info: rotations.append(info['best_rotation_deg'])

    e.close()
    if vec: vec.close()
    return {'rewards': rewards, 'heights': heights, 'rotations': rotations}


print('Evaluating PPO...')
ppo_res = full_eval('ppo', n_episodes=10)
print('Evaluating DQN...')
dqn_res = full_eval('dqn', n_episodes=10, n_actions=N_DISCRETE_ACTIONS)

print(f'\n{"="*55}')
print(f'  PPO vs DQN Backflip Baseline Comparison')
print(f'{"="*55}')
for name, res in [('PPO', ppo_res), ('DQN', dqn_res)]:
    r, h, rot = res['rewards'], res['heights'], res['rotations']
    print(f'\n  {name}:')
    print(f'    Episodes:     {len(r)}')
    print(f'    Avg reward:   {np.mean(r):.1f} ± {np.std(r):.1f}')
    if rot:
        print(f'    Avg rotation: {np.mean(rot):.1f}° ± {np.std(rot):.1f}°')
        print(f'    Max rotation: {np.max(rot):.1f}°')
    if h:
        print(f'    Avg height:   {np.mean(h):.3f}m')
        print(f'    Max height:   {np.max(h):.3f}m')
    if not rot and not h:
        print(f'    (no flights detected)')

Evaluating PPO...
Evaluating DQN...

  PPO vs DQN Backflip Baseline Comparison

  PPO:
    Episodes:     10
    Avg reward:   673.8 ± 9.2
    Avg rotation: 239.9° ± 2.3°
    Max rotation: 243.3°
    Avg height:   0.041m
    Max height:   0.057m

  DQN:
    Episodes:     10
    Avg reward:   52.6 ± 28.7
    Avg rotation: 13.6° ± 4.2°
    Max rotation: 25.2°
    Avg height:   -0.003m
    Max height:   0.007m


In [8]:
# @title 8. Record Video (PPO)

import imageio

def record_video(algo='ppo', out='backflip.mp4', max_steps=300, n_actions=64):
    is_dqn = (algo == 'dqn')
    vn_path = f'backflip_{algo}_vecnorm.pkl'

    raw = gym.make('Humanoid-v5', render_mode='rgb_array')
    e = BackflipRewardWrapper(raw, max_steps, JUMP_SCALE, FLIP_SCALE)
    if is_dqn: e = DiscreteActionWrapper(e, n_actions=n_actions)

    def _mk():
        env = gym.make('Humanoid-v5', render_mode=None)
        env = BackflipRewardWrapper(env, max_steps, JUMP_SCALE, FLIP_SCALE)
        if is_dqn: env = DiscreteActionWrapper(env, n_actions=n_actions)
        return env
    dummy = DummyVecEnv([_mk])
    vec = VecNormalize.load(vn_path, dummy) if os.path.exists(vn_path) else None
    if vec: vec.training = False; vec.norm_reward = False

    model = (DQN if is_dqn else PPO).load(f'backflip_{algo}')
    obs, _ = e.reset(); frames = []

    for _ in range(max_steps):
        frames.append(e.render())
        obs_n = vec.normalize_obs(obs) if vec else obs
        act, _ = model.predict(obs_n, deterministic=True)
        obs, _, term, trunc, _ = e.step(act)
        if term or trunc: obs, _ = e.reset()

    e.close()
    if vec: vec.close()
    imageio.mimsave(out, frames, fps=30)
    print(f'Video saved: {out} ({len(frames)} frames)')

    from IPython.display import Video, display as ipy_display
    ipy_display(Video(out, embed=True))

record_video('ppo', 'backflip_ppo.mp4')

Video saved: backflip_ppo.mp4 (300 frames)


In [9]:
# @title 9. Record Video (DQN)

record_video('dqn', 'backflip_dqn.mp4', n_actions=N_DISCRETE_ACTIONS)

Video saved: backflip_dqn.mp4 (300 frames)


In [10]:
# @title 10. (Optional) Download Models

try:
    from google.colab import files
    for f in ['backflip_ppo.zip', 'backflip_ppo_vecnorm.pkl',
              'backflip_dqn.zip', 'backflip_dqn_vecnorm.pkl']:
        if os.path.exists(f): files.download(f)
except ImportError:
    print('Not on Colab — files in current directory')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>